# install

In [1]:
! pip install lightning
! pip install torchview
! pip install torchmetrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 827.9/827.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 64.2 MB/s eta 0:00:00


# import

In [2]:
# ===============================
# PyTorch Core Modules
# ===============================
import torch  # Core tensor operations
from torch import Tensor
import torch.nn as nn  # Neural network layers
import torch.nn.functional as F  # Functional API for activations, losses, etc.
import torch.optim as optim  # Optimizers
from torch.utils.data import Dataset, DataLoader, random_split  # Dataset utilities

# ===============================
# PyTorch Ecosystem
# ===============================
import pytorch_lightning as pl  # High-level training framework
import torchmetrics  # Model evaluation metrics
from torchview import draw_graph  # Model visualization
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor # Callbacks

# ===============================
# TorchVision - For image data
# ===============================
import torchvision  # Pretrained models and datasets
import torchvision.transforms as transforms  # Image preprocessing utilities
import torchvision.models as models
from torchvision.io import read_image, decode_image
from torchvision.models import resnet50, ResNet50_Weights, MobileNet_V3_Large_Weights

# ===============================
# Scikit-learn - Data generation and preprocessing
# ===============================
from sklearn.datasets import make_regression  # Synthetic regression data
from sklearn.model_selection import train_test_split  # Train/test splitting
from sklearn.preprocessing import MinMaxScaler  # Feature scaling

# ===============================
# Visualization
# ===============================
import matplotlib.pyplot as plt  # Plotting utilities

# ===============================
# Numerical Computing
# ===============================
import numpy as np  # Array operations
import math  # Basic mathematical functions

from PIL import Image
import requests
import io
import json

import os

# Dataset

## Load

In [3]:
import kagglehub

path = kagglehub.dataset_download("yudhaislamisulistya/plants-type-datasets")

print("Path to dataset files:", path)
# %ls -R {path}

100%|██████████| 937M/937M [00:11<00:00, 83.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yudhaislamisulistya/plants-type-datasets/versions/16


## PlantDataset


In [4]:
from torchvision.io import read_image

class PlantDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []
        self.labels = []
        self.class_to_idx = {}
        idx = 0
        for class_name in os.listdir(root_dir):
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                self.class_to_idx[class_name] = idx
                for image_file in os.listdir(class_dir):
                    if image_file.endswith('.jpg') or image_file.endswith('.png'):
                        self.image_files.append(os.path.join(class_dir, image_file))
                        self.labels.append(idx)
                idx += 1

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = read_image(img_path).float() / 255.0 # Read as float and scale to [0, 1]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

## transforms



In [5]:
from torchvision import transforms

train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    # transforms.ToTensor(), # Remove ToTensor as read_image already returns a tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((32, 32)), # Changed from 128x128 to 32x32
    # transforms.ToTensor(), # Remove ToTensor as read_image already returns a tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((32, 32)), # Changed from
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## DataLoader


In [6]:
train_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Train_Set_Folder'
val_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Validation_Set_Folder'
test_root_dir = f'{path}/split_ttv_dataset_type_of_plants/Test_Set_Folder'

train_dataset = PlantDataset(root_dir=train_root_dir, transform=train_transforms)
val_dataset = PlantDataset(root_dir=val_root_dir, transform=val_transforms)
test_dataset = PlantDataset(root_dir=test_root_dir, transform=test_transforms)

batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of batches in training DataLoader: {len(train_dataloader)}")
print(f"Number of batches in validation DataLoader: {len(val_dataloader)}")
print(f"Number of batches in test DataLoader: {len(test_dataloader)}")

Number of batches in training DataLoader: 375
Number of batches in validation DataLoader: 48
Number of batches in test DataLoader: 47


# efficientnet b1

## wrapper

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchmetrics
import pytorch_lightning as pl
from torchvision.models import EfficientNet_B1_Weights

class EfficientNetB1PlantClassifierLightningModule(pl.LightningModule):
    def __init__(self, num_classes, learning_rate=0.001):
        super().__init__()
        self.save_hyperparameters()

        weights = EfficientNet_B1_Weights.DEFAULT
        self.model = torchvision.models.efficientnet_b1(weights=weights)

        # for param in self.model.features.parameters():
        #     param.requires_grad = False

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, num_classes)

        self.criterion = nn.CrossEntropyLoss()
        self.accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.precision = torchmetrics.Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.recall = torchmetrics.Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.f1_score = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', self.accuracy(outputs, labels), prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', self.accuracy(outputs, labels), prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        self.log('test_loss', loss)
        self.log('test_acc', self.accuracy(outputs, labels))
        self.log('test_precision', self.precision(outputs, labels))
        self.log('test_recall', self.recall(outputs, labels))
        self.log('test_f1', self.f1_score(outputs, labels))
        return loss


    def configure_optimizers(self):
        # optimizer = optim.Adam(filter(lambda p: p.requires_grad, self.parameters()), lr=self.hparams.learning_rate)
        optimizer = optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
        return {"optimizer": optimizer}

    # def on_train_epoch_start(self):
    #     if self.current_epoch == 1:
    #         for param in self.model.features.parameters():
    #             param.requires_grad = True
    #         self.trainer.optimizers[0] = optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    #         print(f"Epoch {self.current_epoch}: Unfreezing EfficientNetB1 features and reconfiguring optimizer.")


num_classes = len(train_dataset.class_to_idx)
efficientnet_b1_lightning_model = EfficientNetB1PlantClassifierLightningModule(num_classes=num_classes)

print("✅ EfficientNet B1 Plant Classifier defined.")
print(efficientnet_b1_lightning_model)

Downloading: "https://download.pytorch.org/models/efficientnet_b1-c27df63c.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b1-c27df63c.pth


100%|██████████| 30.1M/30.1M [00:00<00:00, 138MB/s]


✅ EfficientNet B1 Plant Classifier defined.
EfficientNetB1PlantClassifierLightningModule(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1

## train

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    dirpath='checkpoints/',
    filename='best-efficientnet-b1-model',
    save_top_k=1,
    verbose=True
)

early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    mode='min',
    patience=3,
    verbose=True
)

lr_monitor_callback = LearningRateMonitor(logging_interval='step')

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks.progress import TQDMProgressBar

# Make sure the bar updates properly in Kaggle
progress_bar = TQDMProgressBar(refresh_rate=1)

# Optional: enable detailed logs
logger = CSVLogger(save_dir="logs/", name="efficientnet_b0")

trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto',
    devices=1,
    callbacks=[checkpoint_callback, early_stopping_callback, lr_monitor_callback, progress_bar],
    logger=logger,
    enable_progress_bar=True,
    enable_model_summary=True,
    log_every_n_steps=1
)

print("Starting EfficientNet B1 Plant Dataset training with callbacks...")
trainer.fit(efficientnet_b1_lightning_model, train_dataloader, val_dataloader)

print("\nStarting EfficientNet B1 Plant Dataset testing...")

test_results_b1 = trainer.test(efficientnet_b1_lightning_model, test_dataloader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Starting EfficientNet B1 Plant Dataset training with callbacks...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name      | Type                | Params | Mode 
----------------------------------------------------------
0 | model     | EfficientNet        | 6.6 M  | train
1 | criterion | CrossEntropyLoss    | 0      | train
2 | accuracy  | MulticlassAccuracy  | 0      | train
3 | precision | MulticlassPrecision | 0      | train
4 | recall    | MulticlassRecall    | 0      | train
5 | f1_score  | MulticlassF1Score   | 0      | train
----------------------------------------------------------
6.6 M     Trainable params
0         Non-trainable params
6.6 M     Total params
26.206    Total estimated model params size (MB)
478       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved. New best score: 0.896
INFO:pytorch_lightning.utilities.rank_zero:Epoch 0, global step 375: 'val_acc' reached 0.71155 (best 0.71155), saving model to '/content/checkpoints/best-efficientnet-b1-model.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.264 >= min_delta = 0.0. New best score: 0.632
INFO:pytorch_lightning.utilities.rank_zero:Epoch 1, global step 750: 'val_acc' reached 0.79769 (best 0.79769), saving model to '/content/checkpoints/best-efficientnet-b1-model.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Metric val_loss improved by 0.257 >= min_delta = 0.0. New best score: 0.375
INFO:pytorch_lightning.utilities.rank_zero:Epoch 2, global step 1125: 'val_acc' reached 0.87327 (best 0.87327), saving model to '/content/checkpoints/best-efficientnet-b1-model.ckpt' as top 1


## save .pt

In [ ]:
# after training
# trainer.fit(efficientnet_b1_lightning_model, train_dataloader, val_dataloader)

# path to the best checkpoint from your ModelCheckpoint callback
best_ckpt_path = checkpoint_callback.best_model_path
print(f"Best checkpoint path: {best_ckpt_path}")

# load model from checkpoint
model = efficientnet_b1_lightning_model.__class__.load_from_checkpoint(best_ckpt_path)

# optional: move to CPU for saving (especially if training was on GPU)
import torch
model = model.to(torch.device("cpu"))
model.eval()

# save the full model (architecture + weights) as .pt
torch.save(model, "model_best_full.pt")

# OR: save only the state_dict (weights only)
torch.save(model.state_dict(), "model_best_weights.pt")

print("Saved model files: model_best_full.pt and model_best_weights.pt")

## Save stable class index mapping
This cell (do not run until after `train_dataset` exists) writes `model/class_map.json`.
Reason: `PlantDataset` builds `class_to_idx` from directory listing order (`os.listdir`), which can vary across environments. Persisting an explicit idx→label mapping guarantees consistent labels in FastAPI.

Steps when you are ready:
1. Ensure `train_dataset` is already created.
2. Run the code cell below once.
3. Commit the generated `model/class_map.json` file with the model artifacts.

If you later add/remove classes, regenerate this file.

In [ ]:
import os, json

# Ensure model directory exists
os.makedirs("model", exist_ok=True)

# Guard: only proceed if train_dataset exists
if 'train_dataset' not in globals():
    raise NameError("train_dataset not defined. Create dataset before running this cell.")

class_to_idx = train_dataset.class_to_idx  # {class_name: idx}
idx_to_class = {str(v): k for k, v in class_to_idx.items()}  # JSON keys as strings

with open("model/class_map.json", "w") as f:
    json.dump(idx_to_class, f, indent=2)

print("Saved model/class_map.json with", len(idx_to_class), "classes.")